In [0]:
%pip install --upgrade typing_extensions databricks-langchain langchain sentencepiece transformers==4.45.2 torch IndicTransToolkit scikit-learn sentence-transformers

In [0]:
dbutils.widgets.text("hf_token_widget", "", "Enter HuggingFace Token")

# 2. Pull the token from the widget and login
from huggingface_hub import login

token = dbutils.widgets.get("hf_token_widget")

if token:
    login(token=token)
    print("✓ Successfully authenticated with HuggingFace")
else:
    print("⚠ Please enter your token in the text box at the top of the page.")

In [0]:
from databricks_langchain import ChatDatabricks
from sentence_transformers import SentenceTransformer

# 1. Initialize the LLM (Replace 'databricks-llama-3-70b' with your actual endpoint name)
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.1)

# 2. Initialize Embeddings on CPU
embed_model = SentenceTransformer('l3cube-pune/indic-sentence-bert-nli', device="cpu")

In [0]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
ip = IndicProcessor(inference=True)

def translate_to_hindi(text):
    src_lang, tgt_lang = "eng_Latn", "hin_Deva"
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)
    inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt")
    
    with torch.no_grad():
        generated_tokens = model.generate(**inputs, num_beams=5, max_length=512)
    
    translations = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return ip.postprocess_batch(translations, lang=tgt_lang)[0]

In [0]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_legal_context(query):
    # Vectorize the query using your existing embed_model
    query_vec = embed_model.encode([query])[0]
    
    # Load only the BNS table from your catalog
    print("🔎 Searching Legal Database (BNS)...")
    bns_df = spark.table("workspace.default.bns_vector_table").toPandas()
    
    # Calculate similarity
    bns_vecs = np.array(bns_df['Embedding_Vector'].tolist())
    bns_scores = cosine_similarity([query_vec], bns_vecs)[0]
    bns_df['score'] = bns_scores
    
    # Get the top matching legal section
    top_law = bns_df.sort_values(by='score', ascending=False).iloc[0]
    
    # Return context including section name/number for better LLM performance
    context = f"Section {top_law['Section']} ({top_law['Section_name']}): {top_law['Text_Chunk']}"
    return context

In [0]:
# Input
user_query = "I am a laborer and my employer is not paying me my wages."

# Step 1: Get Legal Context only (pass empty state since we don't need schemes)
legal_context, _ = get_context(user_query, "")

# Step 2: Construct the Augmented Prompt (Focused on Law)
prompt = f"""You are 'Nyaya-Sahayak', a specialized Indian Legal Assistant. 
Your goal is to explain the law to citizens in simple, empathetic terms.

Use the LEGAL CONTEXT provided below to answer the user's question. 
If the context isn't relevant, advise them to consult a legal professional.

LEGAL CONTEXT (Bharatiya Nyaya Sanhita):
{legal_context}

USER QUESTION: {user_query}

ANSWER:"""

# Step 3: Call the LLM (Databricks Llama-3)
response = llm.invoke(prompt)
english_text = response.content

print(f"--- AI Legal Response (English) ---\n{english_text}")

# Step 4: Translate (Using your notebook's existing logic)
try:
    hindi_output = translate_to_hindi(english_text)
    print(f"\n--- AI Legal Response (Hindi) ---\n{hindi_output}")
except Exception as e:
    print(f"Translation Error: {e}")